# dense_embedding_exp1

본 노트북은 dense embedding cosine similarity 분석을 위한 exp1 데이터셋을 생성한다.

## exp1 분석 방식

exp1은 기준 문장과 공시 텍스트를 **문서 단위**로 비교하는 방식이다.

즉, 기업 1개와 사업연도 1개를 하나의 분석 단위로 보고, 해당 기업의 사업보고서 중 II, IV, VI, VIII, XI 섹션을 모두 합쳐 하나의 문서로 만든다.

## exp1의 핵심 설계

- 비교 단위: 기업·연도별 문서 전체
- 텍스트 구성: II, IV, VI, VIII, XI 섹션 결합
- 전처리 수준: dense embedding에 맞게 문장 의미를 최대한 보존
- 형태소 분석: 사용하지 않음
- 최종 컬럼: `exp1_document_text`


# 1. 라이브러리 및 경로 설정

In [1]:
# ============================================================
# 1. 라이브러리 및 경로 설정
# ============================================================

import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def find_project_root() -> Path:
    """
    현재 작업 위치에서 data 폴더가 있는 프로젝트 루트를 찾는다.
    """
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data").exists():
            return candidate
    return Path.cwd()


PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ANALYSIS_DIR = PROCESSED_DIR / "analysis"

REPORT_JSON_DIR = PROCESSED_DIR / "dart_full_reports"
COMPANY_MASTER_PATH = RAW_DIR / "company_master.csv"

DENSE_DIR = ANALYSIS_DIR / "dense_embedding_compare"
DENSE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPORT_JSON_DIR:", REPORT_JSON_DIR)
print("COMPANY_MASTER_PATH:", COMPANY_MASTER_PATH)
print("DENSE_DIR:", DENSE_DIR)

PROJECT_ROOT: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final
REPORT_JSON_DIR: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\dart_full_reports
COMPANY_MASTER_PATH: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\raw\company_master.csv
DENSE_DIR: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\analysis\dense_embedding_compare


# 2. 분석 대상 섹션 설정

In [2]:
# ============================================================
# 2. 분석 대상 섹션 설정
# ============================================================

selected_sections = {
    "II": "II. 사업의 내용",
    "IV": "IV. 이사의 경영진단 및 분석의견",
    "VI": "VI. 이사회 등 회사의 기관에 관한 사항",
    "VIII": "VIII. 임원 및 직원 등에 관한 사항",
    "XI": "XI. 그 밖에 투자자 보호를 위하여 필요한 사항",
}

selected_romans = list(selected_sections.keys())

roman_replace_map = {
    "Ⅰ": "I",
    "Ⅱ": "II",
    "Ⅲ": "III",
    "Ⅳ": "IV",
    "Ⅴ": "V",
    "Ⅵ": "VI",
    "Ⅶ": "VII",
    "Ⅷ": "VIII",
    "Ⅸ": "IX",
    "Ⅹ": "X",
    "Ⅺ": "XI",
    "Ⅻ": "XII",
}


def normalize_roman_text(text):
    """
    특수 로마숫자를 일반 영문 로마숫자로 변환한다.
    """
    text = str(text)
    for k, v in roman_replace_map.items():
        text = text.replace(k, v)
    return text


def extract_selected_section_roman(lineage="", text=""):
    """
    lineage 또는 text 앞부분에서 II, IV, VI, VIII, XI 섹션 번호를 추출한다.
    """
    raw = normalize_roman_text(str(lineage))

    if raw.strip() == "" or raw.lower() == "nan":
        raw = normalize_roman_text(str(text)[:1000])

    # 긴 로마숫자부터 먼저 탐색
    pattern = r"\b(VIII|VI|IV|XI|II)\b\.?"
    match = re.search(pattern, raw)

    if match:
        return match.group(1)

    return "미분류"


print("분석 대상 섹션")
for k, v in selected_sections.items():
    print(k, ":", v)

분석 대상 섹션
II : II. 사업의 내용
IV : IV. 이사의 경영진단 및 분석의견
VI : VI. 이사회 등 회사의 기관에 관한 사항
VIII : VIII. 임원 및 직원 등에 관한 사항
XI : XI. 그 밖에 투자자 보호를 위하여 필요한 사항


# 3. DART JSON에서 핵심 섹션 텍스트 추출

In [3]:
# ============================================================
# 3. DART JSON에서 핵심 섹션 텍스트 추출
# ============================================================

json_files = sorted(REPORT_JSON_DIR.glob("*.json"))

print("JSON 파일 수:", len(json_files))

if len(json_files) == 0:
    raise FileNotFoundError("dart_full_reports 폴더에 json 파일이 없습니다.")


def flatten_records(obj):
    """
    JSON 내부에서 text와 lineage 후보를 재귀적으로 추출한다.
    """
    records = []

    text_keys = [
        "text",
        "full_text",
        "unit_text",
        "section_text",
        "content",
        "paragraph",
        "passage",
    ]

    lineage_keys = [
        "lineage",
        "section",
        "section_title",
        "title",
        "heading",
        "chapter",
        "toc",
    ]

    if isinstance(obj, dict):
        keys = set(obj.keys())
        has_text = any(k in keys for k in text_keys)

        if has_text:
            rec = {}

            for k in text_keys:
                if k in obj:
                    rec["text"] = obj.get(k)
                    rec["text_source_key"] = k
                    break

            for k in lineage_keys:
                if k in obj:
                    rec["lineage"] = obj.get(k)
                    rec["lineage_source_key"] = k
                    break

            if "lineage" not in rec:
                rec["lineage"] = ""
                rec["lineage_source_key"] = ""

            records.append(rec)

        for v in obj.values():
            records.extend(flatten_records(v))

    elif isinstance(obj, list):
        for item in obj:
            records.extend(flatten_records(item))

    return records


rows = []

for path in json_files:
    file_name = path.name

    # 예: 005930_2024.json
    m = re.match(r"(\d{6})_(\d{4})\.json", file_name)

    if m:
        stock_code = m.group(1)
        fiscal_year = int(m.group(2))
    else:
        stock_code = ""
        fiscal_year = np.nan

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    records = flatten_records(data)

    for i, rec in enumerate(records):
        lineage = rec.get("lineage", "")

        if isinstance(lineage, list):
            lineage = " > ".join([str(x) for x in lineage])
        else:
            lineage = str(lineage)

        text = str(rec.get("text", ""))

        section_roman = extract_selected_section_roman(
            lineage=lineage,
            text=text
        )

        if section_roman not in selected_romans:
            continue

        rows.append({
            "source_file": file_name,
            "stock_code": str(stock_code).zfill(6),
            "fiscal_year": int(fiscal_year),
            "esg_year": int(fiscal_year) + 1,
            "row_order": i + 1,
            "section_roman": section_roman,
            "section_clean": selected_sections[section_roman],
            "lineage": lineage,
            "text": text,
            "text_chars": len(text),
            "text_source_key": rec.get("text_source_key", ""),
            "lineage_source_key": rec.get("lineage_source_key", ""),
        })


section_text_df = pd.DataFrame(rows)

print("선택 섹션 텍스트 행 수:", len(section_text_df))

if len(section_text_df) == 0:
    raise ValueError("II, IV, VI, VIII, XI 섹션이 추출되지 않았습니다.")

print("분석 기업-연도 수:", section_text_df[["stock_code", "fiscal_year"]].drop_duplicates().shape[0])
print("\n섹션별 행 수")
print(section_text_df["section_clean"].value_counts())

display(section_text_df.head())

JSON 파일 수: 381
선택 섹션 텍스트 행 수: 176
분석 기업-연도 수: 176

섹션별 행 수
section_clean
II. 사업의 내용                     142
VIII. 임원 및 직원 등에 관한 사항          18
VI. 이사회 등 회사의 기관에 관한 사항          7
IV. 이사의 경영진단 및 분석의견              5
XI. 그 밖에 투자자 보호를 위하여 필요한 사항      4
Name: count, dtype: int64


,source_file,stock_code,fiscal_year,esg_year,row_order,section_roman,section_clean,lineage,text,text_chars,text_source_key,lineage_source_key
0,000020_2023.json,000020,2023,2024,1,II,II. 사업의 내용,,감사보고서 \n4.1 \n동화약품 주식회사 \n\n00469373 \n1000000...,441278,full_text,
1,000020_2024.json,000020,2024,2025,1,II,II. 사업의 내용,,연결감사보고서 \n6.0 \n동화약품 주식회사 \n\n00469373 \n10000...,452670,full_text,
2,000050_2022.json,000050,2022,2023,1,II,II. 사업의 내용,,사업보고서 \n5.0 \n(주)경방 \n\nC \nA \n\n사 업 보 고 서 \n...,351848,full_text,
3,000050_2023.json,000050,2023,2024,1,II,II. 사업의 내용,,감사보고서 \n4.1 \n주식회사 경방 \n\n00260198 \n100000000...,373273,full_text,
4,000050_2024.json,000050,2024,2025,1,II,II. 사업의 내용,,연결감사보고서 \n6.0 \n주식회사 경방 \n\n00260198 \n1000000...,414575,full_text,


# 4. 기업-연도별 문서 단위 텍스트 생성

- 같은 기업·연도에 속하는 II, IV, VI, VIII, XI 섹션 텍스트를 하나로 합친다.
- exp1에서는 이 결합 텍스트 전체가 dense embedding cosine 비교 대상이 된다.
- 결과 컬럼은 `raw_document_text`이다.

즉, exp1의 비교 단위는 다음과 같다.

`기준 문장 ↔ 기업·연도별 전체 문서`

In [4]:
# ============================================================
# 4. 기업·연도별 문서 단위 텍스트 생성
# ============================================================

section_order_map = {
    "II": 2,
    "IV": 4,
    "VI": 6,
    "VIII": 8,
    "XI": 11,
}

section_text_df["section_order"] = section_text_df["section_roman"].map(section_order_map)

section_text_df = section_text_df.sort_values(
    ["stock_code", "fiscal_year", "section_order", "row_order"]
)

document_df = (
    section_text_df
    .groupby(["stock_code", "fiscal_year", "esg_year"], as_index=False)
    .agg(
        raw_document_text=("text", lambda x: "\n".join(x.astype(str))),
        included_sections=("section_clean", lambda x: " | ".join(sorted(set(x)))),
        section_count=("section_clean", "nunique"),
        document_text_chars=("text_chars", "sum"),
        document_text_rows=("text", "count"),
    )
)

document_df["analysis_unit"] = (
    document_df["stock_code"] + "x" + document_df["fiscal_year"].astype(str)
)

print("기업-연도 문서 단위 데이터 수:", len(document_df))
print("\n포함 섹션 개수 분포")
print(document_df["section_count"].value_counts().sort_index())

display(document_df.head())

기업-연도 문서 단위 데이터 수: 176

포함 섹션 개수 분포
section_count
1    176
Name: count, dtype: int64


,stock_code,fiscal_year,esg_year,raw_document_text,included_sections,section_count,document_text_chars,document_text_rows,analysis_unit
0,000020,2023,2024,감사보고서 \n4.1 \n동화약품 주식회사 \n\n00469373 \n1000000...,II. 사업의 내용,1,441278,1,000020x2023
1,000020,2024,2025,연결감사보고서 \n6.0 \n동화약품 주식회사 \n\n00469373 \n10000...,II. 사업의 내용,1,452670,1,000020x2024
2,000050,2022,2023,사업보고서 \n5.0 \n(주)경방 \n\nC \nA \n\n사 업 보 고 서 \n...,II. 사업의 내용,1,351848,1,000050x2022
3,000050,2023,2024,감사보고서 \n4.1 \n주식회사 경방 \n\n00260198 \n100000000...,II. 사업의 내용,1,373273,1,000050x2023
4,000050,2024,2025,연결감사보고서 \n6.0 \n주식회사 경방 \n\n00260198 \n1000000...,II. 사업의 내용,1,414575,1,000050x2024


# 5. ESG 등급 데이터 병합

In [5]:
# ============================================================
# 5. ESG 등급 데이터 병합
# ============================================================

company_master = pd.read_csv(COMPANY_MASTER_PATH, dtype={"stock_code": str})
company_master.columns = [str(c).strip() for c in company_master.columns]

company_master["stock_code"] = (
    company_master["stock_code"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(6)
)

company_master["fiscal_year"] = company_master["fiscal_year"].astype(int)

if "esg_year" in company_master.columns:
    company_master["esg_year"] = company_master["esg_year"].astype(int)
else:
    company_master["esg_year"] = company_master["fiscal_year"] + 1

grade_map = {
    "D": 0,
    "C": 1,
    "B": 2,
    "B+": 3,
    "A": 4,
    "A+": 5,
    "S": 6,
}

if "esg_grade_num" not in company_master.columns:
    company_master["esg_grade"] = company_master["esg_grade"].astype(str).str.strip()
    company_master["esg_grade_num"] = company_master["esg_grade"].map(grade_map)

grade_cols = [
    "stock_code",
    "fiscal_year",
    "esg_year",
    "esg_grade",
    "esg_grade_num",
]

for col in ["company_name", "회사이름", "e_grade", "s_grade", "g_grade"]:
    if col in company_master.columns:
        grade_cols.append(col)

analysis_df = pd.merge(
    document_df,
    company_master[grade_cols].drop_duplicates(),
    on=["stock_code", "fiscal_year", "esg_year"],
    how="left"
)

print("병합 후 데이터 수:", len(analysis_df))
print("ESG 등급 결측 수:", analysis_df["esg_grade"].isna().sum())

display(analysis_df.head())

병합 후 데이터 수: 176
ESG 등급 결측 수: 0


,stock_code,fiscal_year,esg_year,raw_document_text,included_sections,section_count,document_text_chars,document_text_rows,analysis_unit,esg_grade,esg_grade_num,company_name,e_grade,s_grade,g_grade
0,000020,2023,2024,감사보고서 \n4.1 \n동화약품 주식회사 \n\n00469373 \n1000000...,II. 사업의 내용,1,441278,1,000020x2023,C,1,동화약품,B,B,C
1,000020,2024,2025,연결감사보고서 \n6.0 \n동화약품 주식회사 \n\n00469373 \n10000...,II. 사업의 내용,1,452670,1,000020x2024,C,1,동화약품,B,C,C
2,000050,2022,2023,사업보고서 \n5.0 \n(주)경방 \n\nC \nA \n\n사 업 보 고 서 \n...,II. 사업의 내용,1,351848,1,000050x2022,C,1,경방,C,D,C
3,000050,2023,2024,감사보고서 \n4.1 \n주식회사 경방 \n\n00260198 \n100000000...,II. 사업의 내용,1,373273,1,000050x2023,C,1,경방,C,D,C
4,000050,2024,2025,연결감사보고서 \n6.0 \n주식회사 경방 \n\n00260198 \n1000000...,II. 사업의 내용,1,414575,1,000050x2024,C,1,경방,C,D,C


# 6. 문서 단위 dense embedding용 기본 정제

dense embedding 방식에서는 문장 의미 보존이 중요하므로 형태소 분석은 적용하지 않는다.

이 단계에서는 다음 정도의 기본 정제만 수행한다.

- 공백 정리
- 줄바꿈 정리
- 깨진 특수문자 일부 제거
- 과도한 기호 제거
- 문장 구조는 최대한 유지

In [6]:
# ============================================================
# 6. 문서 단위 dense embedding용 기본 정제
# ============================================================

def clean_for_dense_document(text):
    """
    dense embedding용 기본 정제 함수.
    문장 의미 보존을 위해 과도한 전처리는 하지 않는다.
    """
    text = str(text)

    # HTML 태그처럼 보이는 것 제거
    text = re.sub(r"<[^>]+>", " ", text)

    # 분석에 크게 필요 없는 특수문자 제거
    # 단, 문장부호 일부는 의미 흐름을 위해 유지
    text = re.sub(r"[^\w가-힣\s.,;:!?%()\-]", " ", text)

    # 로마숫자 특수문자 정리
    text = normalize_roman_text(text)

    # 공백 정리
    text = re.sub(r"\s+", " ", text).strip()

    return text


analysis_df["exp1_document_text"] = analysis_df["raw_document_text"].apply(
    clean_for_dense_document
)

analysis_df["exp1_document_text_chars"] = (
    analysis_df["exp1_document_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

print("문서 단위 정제 텍스트 생성 완료")

display(
    analysis_df[[
        "analysis_unit",
        "raw_document_text",
        "exp1_document_text",
        "exp1_document_text_chars"
    ]].head(2)
)

문서 단위 정제 텍스트 생성 완료


,analysis_unit,raw_document_text,exp1_document_text,exp1_document_text_chars
0,000020x2023,감사보고서 \n4.1 \n동화약품 주식회사 \n\n00469373 \n1000000...,감사보고서 4.1 동화약품 주식회사 00469373 100000000000 2024...,402729
1,000020x2024,연결감사보고서 \n6.0 \n동화약품 주식회사 \n\n00469373 \n10000...,연결감사보고서 6.0 동화약품 주식회사 00469373 100000000000 61...,413409


# 7. exp1 데이터 점검

- 기업·연도별 문서 수를 확인한다.
- 포함된 섹션 개수를 확인한다.
- 원문 길이와 정제 후 길이를 비교한다.
- 샘플을 직접 확인하여 문장 의미가 과도하게 손실되지 않았는지 확인한다.

In [7]:
# ============================================================
# 7. exp1 데이터 점검
# ============================================================

print("전체 데이터 수:", len(analysis_df))
print("기업 수:", analysis_df["stock_code"].nunique())
print("사업연도 분포:")
print(analysis_df["fiscal_year"].value_counts().sort_index())

print("\n포함 섹션 개수 분포:")
print(analysis_df["section_count"].value_counts().sort_index())

length_summary = analysis_df[
    ["document_text_chars", "exp1_document_text_chars"]
].describe().T

display(length_summary)

sample_cols = [
    "analysis_unit",
    "company_name" if "company_name" in analysis_df.columns else "stock_code",
    "fiscal_year",
    "esg_year",
    "esg_grade",
    "included_sections",
    "exp1_document_text",
]

sample_cols = [c for c in sample_cols if c in analysis_df.columns]

display(
    analysis_df[sample_cols].head(3)
)

전체 데이터 수: 176
기업 수: 100
사업연도 분포:
fiscal_year
2022    65
2023    53
2024    58
Name: count, dtype: int64

포함 섹션 개수 분포:
section_count
1    176
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
document_text_chars,176.0,583974.693182,363608.861480,141003.0,379774.25,484161.0,658115.0,3242647.0
exp1_document_text_chars,176.0,527367.130682,324463.906126,128966.0,342955.50,440060.5,600167.5,2862522.0


,analysis_unit,company_name,fiscal_year,esg_year,esg_grade,included_sections,exp1_document_text
0,000020x2023,동화약품,2023,2024,C,II. 사업의 내용,감사보고서 4.1 동화약품 주식회사 00469373 100000000000 2024...
1,000020x2024,동화약품,2024,2025,C,II. 사업의 내용,연결감사보고서 6.0 동화약품 주식회사 00469373 100000000000 61...
2,000050x2022,경방,2022,2023,C,II. 사업의 내용,사업보고서 5.0 (주)경방 C A 사 업 보 고 서 (제138기) 사업연도 202...


# 8. exp1 결과 저장

In [8]:
# ============================================================
# 8. exp1 결과 저장
# ============================================================

output_path = DENSE_DIR / "dense_embedding_exp1_document_unit.csv"

save_cols = []

for col in [
    "analysis_unit",
    "stock_code",
    "company_name",
    "회사이름",
    "fiscal_year",
    "esg_year",
    "esg_grade",
    "esg_grade_num",
    "included_sections",
    "section_count",
    "document_text_chars",
    "document_text_rows",
    "raw_document_text",
    "exp1_document_text",
    "exp1_document_text_chars",
]:
    if col in analysis_df.columns and col not in save_cols:
        save_cols.append(col)

analysis_df[save_cols].to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path)
print("저장 컬럼:")
print(save_cols)

저장 완료: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\analysis\dense_embedding_compare\dense_embedding_exp1_document_unit.csv
저장 컬럼:
['analysis_unit', 'stock_code', 'company_name', 'fiscal_year', 'esg_year', 'esg_grade', 'esg_grade_num', 'included_sections', 'section_count', 'document_text_chars', 'document_text_rows', 'raw_document_text', 'exp1_document_text', 'exp1_document_text_chars']
